In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # NB_07 — SHAP Explainer
# MAGIC **Computes per-user factor breakdown → writes to `xscore.gold.score_explanations`**
# MAGIC
# MAGIC This is what makes XScore different from a black box.
# MAGIC Every score comes with a human-readable explanation:
# MAGIC "Your score is 672. Top positive: bill payment streak (+82 pts).
# MAGIC  Top negative: no ITR filed (−45 pts)."
# MAGIC
# MAGIC What this notebook does:
# MAGIC - Loads the SHAP explainer saved by NB_05
# MAGIC - Computes SHAP values for all 50K users
# MAGIC - Converts SHAP values to per-pillar contribution scores
# MAGIC - Generates a human-readable explanation for each user
# MAGIC - Writes to `xscore.gold.score_explanations`
# MAGIC - Joins explanations back to `gold.credit_scores` for dashboard use
# MAGIC
# MAGIC **Depends on:** NB_05 complete
# MAGIC **Runtime:** ~8 minutes
# MAGIC **Next:** NB_08_dashboard (Streamlit app on Databricks Apps)

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 1 — Setup

# COMMAND ----------

%pip install lightgbm shap --quiet

# COMMAND ----------

import shap
import lightgbm as lgb
import numpy as np
import pandas as pd
import pickle
import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

spark.sql("USE CATALOG xscore")
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("✓ Setup complete")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 2 — Load model and data

# COMMAND ----------

# Load feature contract
contract     = json.loads(dbutils.fs.head(
    "/Volumes/xscore/bronze/kaggle_raw/feature_contract.json"
))
FEATURE_COLS = contract["feature_cols"]
LABEL_COL    = contract["label_col"]

# Load SHAP explainer saved by NB_05
with open("/Volumes/xscore/bronze/kaggle_raw/lgbm_v3_explainer.pkl", "rb") as f:
    explainer = pickle.load(f)

print("✓ SHAP explainer loaded")

# Load all users from Gold feature store
gold_pd = (spark.table("xscore.gold.credit_feature_store")
           .select(["user_id", "segment"] + FEATURE_COLS)
           .fillna(0.0)
           .toPandas())

print(f"✓ Loaded {len(gold_pd):,} users for SHAP computation")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 3 — Compute SHAP values for all users

# COMMAND ----------

print("Computing SHAP values for all users...")
print("(This takes ~5 minutes for 50K users)")

X_all = gold_pd[FEATURE_COLS]

# compute_shap=True returns SHAP values directly
shap_values = explainer.shap_values(X_all)

# For binary classification LightGBM returns list [class0, class1]
# We want class 1 (probability of default)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f"✓ SHAP values computed: {sv.shape}")
print(f"  Shape: {sv.shape[0]:,} users × {sv.shape[1]} features")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 4 — Map features to pillars

# COMMAND ----------

# Map each feature to its pillar
# Higher SHAP = feature pushes toward default (bad)
# Lower SHAP = feature pushes away from default (good)

PILLAR_MAP = {
    "p1_bill_ontime_rate"      : "Pillar 1 — Bill payment",
    "p1_avg_days_late"         : "Pillar 1 — Bill payment",
    "p1_severe_late_rate"      : "Pillar 1 — Bill payment",
    "p1_bill_type_diversity"   : "Pillar 1 — Bill payment",
    "p1_payment_trend"         : "Pillar 1 — Bill payment",
    "p2_upi_txn_per_month"     : "Pillar 2 — UPI flow",
    "p2_avg_txn_amount"        : "Pillar 2 — UPI flow",
    "p2_txn_cv"                : "Pillar 2 — UPI flow",
    "p2_failure_rate"          : "Pillar 2 — UPI flow",
    "p2_merchant_diversity"    : "Pillar 2 — UPI flow",
    "owns_land_d"              : "Pillar 3 — Assets",
    "land_acres_capped"        : "Pillar 3 — Assets",
    "owns_vehicle_d"           : "Pillar 3 — Assets",
    "bank_vintage_capped"      : "Pillar 3 — Assets",
    "has_fd_or_rd_d"           : "Pillar 3 — Assets",
    "income_log"               : "Pillar 4 — Income",
    "itr_filed_d"              : "Pillar 4 — Income",
    "gst_registered_d"         : "Pillar 4 — Income",
    "employment_capped"        : "Pillar 4 — Income",
    "jan_dhan_active_d"        : "Pillar 5 — Identity",
    "shg_member_d"             : "Pillar 5 — Identity",
    "shg_months"               : "Pillar 5 — Identity",
    "dbt_months"               : "Pillar 5 — Identity",
    "svanidhi_repaid_d"        : "Pillar 5 — Identity",
    "sim_tenure_months"        : "Pillar 6 — Stability",
    "location_stability"       : "Pillar 6 — Stability",
    "fraud_flag_d"             : "Pillar 6 — Stability",
    "derived_income_to_bills_ratio" : "Derived",
    "derived_digital_engagement"    : "Derived",
    "derived_formal_economy_score"  : "Derived",
    "derived_asset_score"           : "Derived",
}

# SHAP values: negative = pushes AWAY from default = GOOD for XScore
# We negate them so positive = good for the user
SHAP_DF = pd.DataFrame(
    -sv,   # negate: positive now means "helps score"
    columns=FEATURE_COLS
)
SHAP_DF["user_id"] = gold_pd["user_id"].values
SHAP_DF["segment"] = gold_pd["segment"].values

print(f"✓ SHAP DataFrame built: {SHAP_DF.shape}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 5 — Compute per-pillar contribution scores

# COMMAND ----------

# Sum SHAP values within each pillar per user
PILLARS = [
    "Pillar 1 — Bill payment",
    "Pillar 2 — UPI flow",
    "Pillar 3 — Assets",
    "Pillar 4 — Income",
    "Pillar 5 — Identity",
    "Pillar 6 — Stability",
    "Derived",
]

for pillar in PILLARS:
    pillar_features = [f for f, p in PILLAR_MAP.items() if p == pillar]
    col_name = pillar.lower().replace(" ", "_").replace("—", "").replace("  ", "_")
    SHAP_DF[f"shap_{col_name}"] = SHAP_DF[pillar_features].sum(axis=1)

# Normalise pillar scores to a 0–100 contribution scale
pillar_cols = [c for c in SHAP_DF.columns if c.startswith("shap_pillar")]

# Scale: convert raw SHAP sums to point contributions (out of 150 per pillar max)
for col in pillar_cols:
    raw = SHAP_DF[col]
    # Clip to [-2, 2] to prevent extreme values from dominating
    clipped = raw.clip(-2, 2)
    # Scale to [-150, +150] point range
    SHAP_DF[col + "_pts"] = (clipped / 2 * 150).round(1)

print("Per-pillar contribution columns created:")
for col in [c for c in SHAP_DF.columns if c.endswith("_pts")]:
    avg = SHAP_DF[col].mean()
    print(f"  {col:<45} avg={avg:+.1f} pts")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 6 — Top 3 positive and negative factors per user

# COMMAND ----------

def get_top_factors(shap_row, feature_cols, top_n=3):
    """
    Returns top N positive (helpful) and negative (harmful) features
    for a single user, as readable strings.
    """
    # Map feature names to human-readable labels
    READABLE = {
        "p1_bill_ontime_rate"    : "Bill payment consistency",
        "p1_avg_days_late"       : "Average payment delay",
        "p1_severe_late_rate"    : "Severely late bills",
        "p1_bill_type_diversity" : "Bill type variety",
        "p1_payment_trend"       : "Payment improvement trend",
        "p2_upi_txn_per_month"   : "UPI transaction volume",
        "p2_avg_txn_amount"      : "Average UPI amount",
        "p2_txn_cv"              : "Transaction consistency",
        "p2_failure_rate"        : "UPI failure rate",
        "p2_merchant_diversity"  : "Merchant diversity",
        "owns_land_d"            : "Land ownership",
        "land_acres_capped"      : "Land holding size",
        "owns_vehicle_d"         : "Vehicle ownership",
        "bank_vintage_capped"    : "Banking tenure",
        "has_fd_or_rd_d"         : "Fixed/recurring deposit",
        "income_log"             : "Income level",
        "itr_filed_d"            : "ITR filing status",
        "gst_registered_d"       : "GST registration",
        "employment_capped"      : "Employment tenure",
        "jan_dhan_active_d"      : "Jan Dhan account activity",
        "shg_member_d"           : "SHG membership",
        "shg_months"             : "SHG tenure",
        "dbt_months"             : "DBT benefit history",
        "svanidhi_repaid_d"      : "SVANidhi loan repaid",
        "sim_tenure_months"      : "Mobile SIM tenure",
        "location_stability"     : "Location stability",
        "fraud_flag_d"           : "Fraud flag",
        "derived_income_to_bills_ratio": "Income vs bills ratio",
        "derived_digital_engagement"   : "Digital engagement",
        "derived_formal_economy_score" : "Formal economy score",
        "derived_asset_score"          : "Asset strength",
    }

    series = pd.Series(dict(zip(feature_cols, shap_row)))
    top_pos = series.nlargest(top_n)
    top_neg = series.nsmallest(top_n)

    pos_str = " | ".join([
        f"{READABLE.get(k, k)} (+{v:.2f})"
        for k, v in top_pos.items() if v > 0
    ])
    neg_str = " | ".join([
        f"{READABLE.get(k, k)} ({v:.2f})"
        for k, v in top_neg.items() if v < 0
    ])

    return pos_str or "No strong positive factors", neg_str or "No strong negative factors"


print("Computing top factors for all users...")
print("(~2 minutes for 50K users)\n")

shap_matrix = SHAP_DF[FEATURE_COLS].values
pos_factors_list = []
neg_factors_list = []

for i, row in enumerate(shap_matrix):
    pos, neg = get_top_factors(row, FEATURE_COLS)
    pos_factors_list.append(pos)
    neg_factors_list.append(neg)
    if (i + 1) % 10000 == 0:
        print(f"  Processed {i+1:,} users...")

SHAP_DF["top_positive_factors"] = pos_factors_list
SHAP_DF["top_negative_factors"] = neg_factors_list

print(f"✓ Top factors computed for all {len(SHAP_DF):,} users")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 7 — Generate human-readable explanation text

# COMMAND ----------

# Join with credit scores to get xscore value
scores_pd = (spark.table("xscore.gold.credit_scores")
             .select("user_id", "xscore", "score_band")
             .toPandas())

SHAP_DF = SHAP_DF.merge(scores_pd, on="user_id", how="left")

def make_explanation(row):
    """
    Generates a one-sentence human-readable score explanation.
    This is what gets displayed on the bank dashboard and user app.
    """
    score = int(row["xscore"]) if pd.notna(row["xscore"]) else 500
    band  = row.get("score_band", "Fair")

    top_pos = str(row["top_positive_factors"]).split(" | ")[0].split("(")[0].strip()
    top_neg = str(row["top_negative_factors"]).split(" | ")[0].split("(")[0].strip()

    if band == "Excellent":
        opener = f"Strong profile — score {score}."
    elif band == "Good":
        opener = f"Good profile — score {score}."
    elif band == "Fair":
        opener = f"Developing profile — score {score}."
    else:
        opener = f"Needs improvement — score {score}."

    if top_pos and top_neg and top_pos != "No strong positive factors":
        return f"{opener} Key strength: {top_pos}. Main opportunity: {top_neg}."
    elif top_pos and top_pos != "No strong positive factors":
        return f"{opener} Key strength: {top_pos}."
    else:
        return f"{opener} Build payment history to improve."

SHAP_DF["explanation_text"] = SHAP_DF.apply(make_explanation, axis=1)

print("Sample explanations:\n")
for _, row in SHAP_DF.sample(5, random_state=42).iterrows():
    print(f"  [{row['segment']:<22}] score={row.get('xscore', '?')}  {row['explanation_text']}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 8 — Write score_explanations to Gold

# COMMAND ----------

# MAGIC %sql
# MAGIC CREATE TABLE IF NOT EXISTS xscore.gold.score_explanations (
# MAGIC   user_id                   STRING NOT NULL,
# MAGIC   segment                   STRING,
# MAGIC   explanation_text          STRING,
# MAGIC   top_positive_factors      STRING,
# MAGIC   top_negative_factors      STRING,
# MAGIC   shap_pillar_1_bill_payment_pts  DOUBLE,
# MAGIC   shap_pillar_2_upi_flow_pts      DOUBLE,
# MAGIC   shap_pillar_3_assets_pts        DOUBLE,
# MAGIC   shap_pillar_4_income_pts        DOUBLE,
# MAGIC   shap_pillar_5_identity_pts      DOUBLE,
# MAGIC   shap_pillar_6_stability_pts     DOUBLE,
# MAGIC   computed_at               TIMESTAMP
# MAGIC )
# MAGIC USING DELTA
# MAGIC COMMENT 'Per-user SHAP-based score explanations — one row per user';

# COMMAND ----------

import pyspark.sql.types as T
from datetime import datetime

# Select the columns we want to write
pts_cols = [c for c in SHAP_DF.columns if c.endswith("_pts") and "pillar" in c]

output_cols = (
    ["user_id", "segment",
     "explanation_text", "top_positive_factors", "top_negative_factors"]
    + pts_cols
)

output_pd = SHAP_DF[output_cols].copy()
output_pd["computed_at"] = datetime.now()

# Rename pts columns to match table schema
rename = {}
for c in pts_cols:
    clean = c.replace(" ", "_").replace("—", "")
    if c != clean:
        rename[c] = clean
output_pd = output_pd.rename(columns=rename)

explanations_spark = spark.createDataFrame(output_pd)

(explanations_spark.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("xscore.gold.score_explanations"))

n = spark.table("xscore.gold.score_explanations").count()
print(f"✓ xscore.gold.score_explanations: {n:,} rows")

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 9 — Verify: show sample score cards

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Show 5 sample score cards with full explanation
# MAGIC SELECT
# MAGIC   s.user_id,
# MAGIC   s.segment,
# MAGIC   c.xscore,
# MAGIC   c.score_band,
# MAGIC   s.explanation_text,
# MAGIC   ROUND(s.shap_pillar_1_bill_payment_pts, 1) AS bill_pts,
# MAGIC   ROUND(s.shap_pillar_2_upi_flow_pts, 1)     AS upi_pts,
# MAGIC   ROUND(s.shap_pillar_4_income_pts, 1)       AS income_pts,
# MAGIC   ROUND(s.shap_pillar_5_identity_pts, 1)     AS identity_pts
# MAGIC FROM xscore.gold.score_explanations s
# MAGIC JOIN xscore.gold.credit_scores c USING (user_id)
# MAGIC ORDER BY c.xscore DESC
# MAGIC LIMIT 5;

# COMMAND ----------

# MAGIC %sql
# MAGIC -- Average pillar contribution by segment
# MAGIC -- Shows which pillars matter most for each borrower type
# MAGIC SELECT
# MAGIC   segment,
# MAGIC   COUNT(*)                                          AS users,
# MAGIC   ROUND(AVG(shap_pillar_1_bill_payment_pts), 1)    AS avg_bill_pts,
# MAGIC   ROUND(AVG(shap_pillar_2_upi_flow_pts), 1)        AS avg_upi_pts,
# MAGIC   ROUND(AVG(shap_pillar_3_assets_pts), 1)          AS avg_asset_pts,
# MAGIC   ROUND(AVG(shap_pillar_4_income_pts), 1)          AS avg_income_pts,
# MAGIC   ROUND(AVG(shap_pillar_5_identity_pts), 1)        AS avg_identity_pts,
# MAGIC   ROUND(AVG(shap_pillar_6_stability_pts), 1)       AS avg_stability_pts
# MAGIC FROM xscore.gold.score_explanations
# MAGIC GROUP BY segment
# MAGIC ORDER BY segment;

# COMMAND ----------
# MAGIC %md
# MAGIC ## Cell 10 — Final summary

# COMMAND ----------

print("=" * 60)
print("  NB_07 SHAP EXPLAINER — COMPLETE")
print("=" * 60)

for tbl in ["credit_scores", "score_explanations", "best_hyperparams"]:
    n = spark.table(f"xscore.gold.{tbl}").count()
    print(f"  ✓  gold.{tbl:<25} {n:>10,} rows")

print()
print("  Every score now has:")
print("    - Human-readable explanation text")
print("    - Top 3 positive and negative factors")
print("    - Per-pillar point contribution (±150 pts each)")
print()
print("  NEXT: NB_08_dashboard")
print("  NB_08 builds the Streamlit app on Databricks Apps")
print("  using gold.credit_scores + gold.score_explanations.")
print("=" * 60)